In [2]:
from google.colab import files
import pandas as pd
import numpy as np
import json

print("bank.csv upload karo:")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH, sep=None, engine='python')
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


bank.csv upload karo:


Saving bank.csv to bank.csv
Loaded: 11162 rows, 17 columns


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [3]:
# QUALITY SCORE ENGINE — cleaning se pehle aur baad dono ke liye
def quality_score(dataframe):
    """Dataset quality score (0-100). Higher = cleaner."""
    total_cells = dataframe.shape[0] * dataframe.shape[1]

    missing = int(dataframe.isna().sum().sum())

    sentinel = int((dataframe['pdays'] == -1).sum()) if 'pdays' in dataframe.columns else 0

    negative = int((dataframe['balance'] < 0).sum()) if 'balance' in dataframe.columns else 0

    outliers = 0
    for col in dataframe.select_dtypes(include=np.number).columns:
        s = dataframe[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        outliers += int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

    penalty = missing + sentinel + negative + outliers
    score = round(100 * (1 - penalty / total_cells), 2)
    details = {'missing': missing, 'sentinel_pdays_-1': sentinel,
               'negative_balance': negative, 'iqr_outliers': outliers}
    return score, details

before_score, before_details = quality_score(df)

print('===== BEFORE CLEANING =====')
print(f'Quality Score: {before_score} / 100')
for k, v in before_details.items():
    print(f'  - {k}: {v}')


===== BEFORE CLEANING =====
Quality Score: 91.84 / 100
  - missing: 0
  - sentinel_pdays_-1: 8324
  - negative_balance: 688
  - iqr_outliers: 6471


In [4]:
# ===== CLEANING ENGINE =====
# Module 1 ne 5 columns flag kiye the — ab wahi 5 fix karenge
cleaning_log = []
df_clean = df.copy()

# FIX 1: pdays sentinel — -1 ka matlab "never contacted" (real NaN banao)
mask = df_clean['pdays'] == -1
n1 = int(mask.sum())
df_clean.loc[mask, 'pdays'] = np.nan
cleaning_log.append({'action': 'sentinel_to_nan', 'column': 'pdays',
                     'detail': '-1 means never contacted before; converted to real missing value',
                     'rows_affected': n1})

# FIX 2: negative balance — bank balance negative nahi ho sakta
mask = df_clean['balance'] < 0
n2 = int(mask.sum())
df_clean.loc[mask, 'balance'] = np.nan
cleaning_log.append({'action': 'invalid_to_nan', 'column': 'balance',
                     'detail': 'impossible negative balances set to missing',
                     'rows_affected': n2})

# FIX 3: outlier capping — extreme values ko IQR fences par lao
CAP_COLS = ['balance', 'duration', 'campaign', 'pdays', 'previous']
n3 = 0
for col in CAP_COLS:
    s = df_clean[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < lo) | (s > hi)
    n3 += int(mask.sum())
    df_clean[col] = s.clip(lo, hi)
cleaning_log.append({'action': 'outlier_capping', 'column': ', '.join(CAP_COLS),
                     'detail': 'extreme values capped at IQR fences (winsorization)',
                     'rows_affected': n3})

print(f"Cleaning actions recorded: {len(cleaning_log)}")
for entry in cleaning_log:
    print(f"  [{entry['action']}] {entry['column']} -> {entry['rows_affected']} rows")

Cleaning actions recorded: 3
  [sentinel_to_nan] pdays -> 8324 rows
  [invalid_to_nan] balance -> 688 rows
  [outlier_capping] balance, duration, campaign, pdays, previous -> 3504 rows


In [5]:
# ===== AFTER SCORE + OUTPUT FILES =====
after_score, after_details = quality_score(df_clean)

print('===== AFTER CLEANING =====')
print(f'Quality Score: {after_score} / 100')
for k, v in after_details.items():
    print(f'  - {k}: {v}')

print()
print('===== BEFORE vs AFTER =====')
print(f'Before: {before_score}  ->  After: {after_score}  (improvement: +{round(after_score - before_score, 2)} points)')

df_clean.to_csv('cleaned_data.csv', index=False)

with open('cleaning_log.json', 'w') as f:
    json.dump({'dataset': 'bank.csv',
               'rows': int(len(df_clean)),
               'quality_score_before': before_score,
               'quality_score_after': after_score,
               'actions': cleaning_log}, f, indent=2)

print()
print('Saved: cleaned_data.csv + cleaning_log.json')
files.download('cleaned_data.csv')
files.download('cleaning_log.json')

===== AFTER CLEANING =====
Quality Score: 95.16 / 100
  - missing: 9012
  - sentinel_pdays_-1: 0
  - negative_balance: 0
  - iqr_outliers: 171

===== BEFORE vs AFTER =====
Before: 91.84  ->  After: 95.16  (improvement: +3.32 points)

Saved: cleaned_data.csv + cleaning_log.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from sklearn.impute import KNNImputer

df_final = df_clean.copy()
imp_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']

imputer = KNNImputer(n_neighbors=5)
df_final[imp_cols] = imputer.fit_transform(df_final[imp_cols])

print('===== KNN IMPUTATION =====')
print(f'pdays: 8324 NaN the -> ab {int(df_final["pdays"].isna().sum())} bache')
print(f'balance: 688 NaN the -> ab {int(df_final["balance"].isna().sum())} bache')
print(f'Total NaN feature columns mein: {int(df_final[imp_cols].isna().sum().sum())}')


===== KNN IMPUTATION =====
pdays: 8324 NaN the -> ab 0 bache
balance: 688 NaN the -> ab 0 bache
Total NaN feature columns mein: 0


In [7]:
dup_key = ['age', 'job', 'marital', 'balance']
groups = df_final.groupby(dup_key).size()
suspects = groups[groups > 1].sort_values(ascending=False)

n_extra = int(suspects.sum() - len(suspects))
print('===== FUZZY DUPLICATE DETECTION =====')
print(f'Suspicious duplicate groups: {len(suspects)}')
print(f'Extra rows (potential duplicates): {n_extra}')
print()
print('Top 5 suspect groups:')
print(suspects.head(5))


===== FUZZY DUPLICATE DETECTION =====
Suspicious duplicate groups: 960
Extra rows (potential duplicates): 1560

Top 5 suspect groups:
age   job         marital  balance
30.0  management  single   4327.0     15
37.0  management  married  4327.0     12
39.0  management  married  0.0        10
31.0  management  single   4327.0     10
34.0  management  married  0.0         9
dtype: int64


In [8]:
from sklearn.preprocessing import StandardScaler

num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

df_scaled = df_final.copy()
df_scaled[num_cols] = StandardScaler().fit_transform(df_scaled[num_cols])
df_encoded = pd.get_dummies(df_scaled, columns=cat_cols, drop_first=True)

print('===== SCALING + ENCODING =====')
print(f'Scaled: {len(num_cols)} numeric columns (StandardScaler)')
print(f'Encoded: {len(cat_cols)} categorical columns (one-hot)')
print(f'ML-ready shape: {df_encoded.shape[0]} rows x {df_encoded.shape[1]} columns')

df_encoded.to_csv('ml_ready_data.csv', index=False)
files.download('ml_ready_data.csv')
print('Saved + downloaded: ml_ready_data.csv')


===== SCALING + ENCODING =====
Scaled: 7 numeric columns (StandardScaler)
Encoded: 9 categorical columns (one-hot)
ML-ready shape: 11162 rows x 43 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved + downloaded: ml_ready_data.csv
